<a href="https://colab.research.google.com/github/shinejeweljaipur/-fifa-eda-project/blob/main/EDA23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
# =============================================================
# WORLD CUP WINNER PREDICTION - EDA + DATA CLEANING (NumPy & Pandas)
# Files: train.csv, test.csv, submission.csv
# =============================================================

import numpy as np
import pandas as pd

# -------------------------------------------------------------
# STEP 0 : LOAD RAW DATA
# -------------------------------------------------------------
TRAIN_PATH      = '/content/train (1).csv'          # apna path yahan set karo
TEST_PATH       = '/content/test (2).csv'
SUBMISSION_PATH = '/content/submission (17).csv'

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
sub_raw   = pd.read_csv(SUBMISSION_PATH)

train = train_raw.copy()       # working copies — raw data untouched
test  = test_raw.copy()
sub   = sub_raw.copy()

TARGET = 'winner'

# -------------------------------------------------------------
# STEP 1 : BASIC STRUCTURE / OVERVIEW
# -------------------------------------------------------------
print("TRAIN shape:", train.shape)
print("TEST  shape:", test.shape)
print("SUBMISSION shape:", sub.shape)

print("\nTrain columns:\n", train.columns.tolist())
print("\nColumns only in train (not test):", set(train.columns) - set(test.columns))

print("\nTrain info:")
train.info()

print("\nTrain head:\n", train.head())
print("\nTest head:\n", test.head())
print("\nSubmission head:\n", sub.head())

print("\nMemory usage train (MB):", train.memory_usage(deep=True).sum() / 1e6)

# -------------------------------------------------------------
# STEP 2 : STATISTICAL SUMMARY
# -------------------------------------------------------------
print("\nNumeric summary (train):\n", train.describe())
print("\nCategorical summary (train):\n", train.describe(include='object'))

num_cols = train.select_dtypes(include=np.number).columns.tolist()
num_cols_no_target = [c for c in num_cols if c != TARGET]
cat_cols = train.select_dtypes(include='object').columns.tolist()
print("\nNumeric columns:", num_cols_no_target)
print("Categorical columns:", cat_cols)

# NumPy-level stats on a key column
print("\nMean fifa_points:", np.mean(train['fifa_points']))
print("Std fifa_points:", np.std(train['fifa_points']))
print("Min/Max fifa_rank:", np.min(train['fifa_rank']), np.max(train['fifa_rank']))
print("Percentiles market_value_million_eur (25/50/75):",
      np.percentile(train['market_value_million_eur'], [25, 50, 75]))

# -------------------------------------------------------------
# STEP 3 : MISSING VALUES
# -------------------------------------------------------------
print("\nMissing values - train:\n", train.isnull().sum()[train.isnull().sum() > 0])
print("\nMissing values - test:\n", test.isnull().sum()[test.isnull().sum() > 0])
print("\n% missing - train:\n", (train.isnull().sum() / len(train) * 100).round(2))

# Handling missing values (only applies if any exist)
for col in num_cols_no_target:
    if train[col].isnull().sum() > 0:
        train[col] = train[col].fillna(train[col].median())
    if col in test.columns and test[col].isnull().sum() > 0:
        test[col] = test[col].fillna(train[col].median())      # use TRAIN median for test too

for col in cat_cols:
    if train[col].isnull().sum() > 0:
        train[col] = train[col].fillna(train[col].mode()[0])
    if col in test.columns and test[col].isnull().sum() > 0:
        test[col] = test[col].fillna(train[col].mode()[0])

print("\nMissing after cleaning - train:", train.isnull().sum().sum())
print("Missing after cleaning - test:", test.isnull().sum().sum())

# -------------------------------------------------------------
# STEP 4 : DUPLICATES
# -------------------------------------------------------------
print("\nDuplicate rows - train:", train.duplicated().sum())
train = train.drop_duplicates()
print("Duplicate team_name entries:", train['team_name'].duplicated().sum())

# -------------------------------------------------------------
# STEP 5 : UNIQUE VALUES / CATEGORICAL EXPLORATION
# -------------------------------------------------------------
print("\nUnique confederations:", train['confederation'].unique())
print("\nConfederation counts:\n", train['confederation'].value_counts())
print("\nHost advantage counts:\n", train['host_advantage'].value_counts())
print("\nTarget (winner) distribution:\n", train[TARGET].value_counts(normalize=True) * 100)

# -------------------------------------------------------------
# STEP 6 : OUTLIER DETECTION (IQR METHOD)
# -------------------------------------------------------------
def detect_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return dataframe[(dataframe[column] < lower) | (dataframe[column] > upper)]

outliers_market_value = detect_outliers_iqr(train, 'market_value_million_eur')
print("\nOutliers in market_value_million_eur:", outliers_market_value.shape[0])

outliers_travel = detect_outliers_iqr(train, 'travel_distance_avg')
print("Outliers in travel_distance_avg:", outliers_travel.shape[0])

# Z-score method (NumPy)
z_scores = (train['fifa_points'] - train['fifa_points'].mean()) / train['fifa_points'].std()
outliers_zscore = train[np.abs(z_scores) > 3]
print("Outliers via z-score in fifa_points:", outliers_zscore.shape[0])

# Capping outliers (winsorization) — example on one column
lower_cap = train['market_value_million_eur'].quantile(0.01)
upper_cap = train['market_value_million_eur'].quantile(0.99)
train['market_value_million_eur'] = np.clip(train['market_value_million_eur'], lower_cap, upper_cap)
test['market_value_million_eur']  = np.clip(test['market_value_million_eur'], lower_cap, upper_cap)

# -------------------------------------------------------------
# STEP 7 : FILTERING / SORTING / GROUPBY
# -------------------------------------------------------------
top_ranked        = train.sort_values(by='fifa_rank').head(10)
high_market_value = train[train['market_value_million_eur'] > train['market_value_million_eur'].quantile(0.90)]
winners_only       = train[train[TARGET] == 1]

print("\nAvg stats by confederation:\n",
      train.groupby('confederation')[['fifa_points', 'avg_player_rating', 'win_rate_last_year']].mean())

print("\nWin rate by host_advantage:\n",
      train.groupby('host_advantage')[TARGET].mean())

agg_summary = train.groupby('confederation').agg(
    avg_fifa_points=('fifa_points', 'mean'),
    avg_market_value=('market_value_million_eur', 'mean'),
    win_rate=(TARGET, 'mean'),
    team_count=('team_name', 'count')
).reset_index().sort_values('win_rate', ascending=False)
print("\nConfederation-wise summary:\n", agg_summary)

# -------------------------------------------------------------
# STEP 8 : CORRELATION WITH TARGET
# -------------------------------------------------------------
corr_with_target = train[num_cols].corr()[TARGET].sort_values(ascending=False)
print("\nCorrelation of features with target (winner):\n", corr_with_target)

# -------------------------------------------------------------
# STEP 9 : FEATURE ENGINEERING (applied consistently to train + test)
# -------------------------------------------------------------
for df_ in [train, test]:
    df_['total_matches']       = df_['wins_last_10_matches'] + df_['losses_last_10_matches'] + df_['draws_last_10_matches']
    df_['win_ratio_10']        = df_['wins_last_10_matches'] / df_['total_matches'].replace(0, np.nan)
    df_['goal_difference_avg'] = df_['goals_scored_avg'] - df_['goals_conceded_avg']
    df_['rank_group']          = pd.cut(df_['fifa_rank'], bins=[0, 10, 30, 60, 100, 300],
                                          labels=['Top10', 'Top30', 'Top60', 'Top100', 'Below100'])
    df_['is_favorite']         = np.where(df_['fifa_rank'] <= 20, 1, 0)
    df_['strength_score']      = (df_['avg_player_rating'] * 0.4 +
                                    df_['recent_form_score'] * 0.3 +
                                    df_['win_rate_last_year'] * 0.3)

# Normalization example (min-max scaling)
for col in ['fifa_points', 'market_value_million_eur']:
    min_val, max_val = train[col].min(), train[col].max()
    train[col + '_scaled'] = (train[col] - min_val) / (max_val - min_val)
    test[col + '_scaled']  = (test[col] - min_val) / (max_val - min_val)   # use TRAIN min/max on test

# -------------------------------------------------------------
# STEP 10 : ENCODING CATEGORICAL VARIABLES
# -------------------------------------------------------------
# One-hot encoding confederation (fit on combined categories to keep columns aligned)
combined_confeds = pd.concat([train['confederation'], test['confederation']]).unique()
train['confederation'] = pd.Categorical(train['confederation'], categories=combined_confeds)
test['confederation']  = pd.Categorical(test['confederation'], categories=combined_confeds)

train_encoded = pd.get_dummies(train, columns=['confederation'], prefix='conf')
test_encoded  = pd.get_dummies(test, columns=['confederation'], prefix='conf')

print("\nEncoded train shape:", train_encoded.shape)

# -------------------------------------------------------------
# STEP 11 : NUMPY ARRAY-LEVEL CHECKS
# -------------------------------------------------------------
target_array = train[TARGET].to_numpy()
print("\nClass balance via NumPy bincount:", np.bincount(target_array))
print("Any team with 100% win rate last 10:", np.any(train['win_ratio_10'].to_numpy() == 1))
print("Teams with fifa_rank in top 5 (indices):", np.where(train['fifa_rank'].to_numpy() <= 5)[0])

# -------------------------------------------------------------
# STEP 12 : FINAL CONSISTENCY CHECKS (train vs test alignment)
# -------------------------------------------------------------
train_feature_cols = [c for c in train.columns if c not in [TARGET, 'team_name', 'country_code']]
test_feature_cols  = [c for c in test.columns if c not in ['team_name', 'country_code']]
print("\nColumns in train but not test:", set(train_feature_cols) - set(test_feature_cols))
print("Columns in test but not train:", set(test_feature_cols) - set(train_feature_cols))

# -------------------------------------------------------------
# STEP 13 : EXPORT CLEANED DATA
# -------------------------------------------------------------
train.to_csv('train_cleaned.csv', index=False)
test.to_csv('test_cleaned.csv', index=False)
print("\nCleaned files exported: train_cleaned.csv, test_cleaned.csv")

# -------------------------------------------------------------
# STEP 14 : SUMMARY REPORT
# -------------------------------------------------------------
with open('cleaning_report.txt', 'w') as f:
    f.write(f"Train shape: {train.shape}\n")
    f.write(f"Test shape: {test.shape}\n")
    f.write(f"Missing values remaining (train): {train.isnull().sum().sum()}\n")
    f.write(f"Duplicate rows removed (train): {train_raw.duplicated().sum()}\n")
    f.write(f"Outliers found in market_value_million_eur: {outliers_market_value.shape[0]}\n")
    f.write(f"Target distribution:\n{train[TARGET].value_counts().to_string()}\n")
    f.write(f"\nTop correlated features with target:\n{corr_with_target.head(10).to_string()}\n")

print("\nEDA + Cleaning complete. Report saved to 'cleaning_report.txt'.")

TRAIN shape: (1000, 26)
TEST  shape: (250, 25)
SUBMISSION shape: (250, 2)

Train columns:
 ['team_name', 'country_code', 'confederation', 'fifa_rank', 'fifa_points', 'wins_last_10_matches', 'losses_last_10_matches', 'draws_last_10_matches', 'win_rate_last_year', 'goals_scored_avg', 'goals_conceded_avg', 'clean_sheets_last_10', 'shots_per_game', 'shots_on_target_ratio', 'avg_player_rating', 'star_players_count', 'market_value_million_eur', 'experience_avg_caps', 'coach_experience_years', 'recent_form_score', 'possession_avg', 'passing_accuracy', 'host_advantage', 'travel_distance_avg', 'climate_similarity_score', 'winner']

Columns only in train (not test): {'winner'}

Train info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   team_name                 1000 non-null   object 
 1   country_code              1000 non-

In [5]:
# =============================================================
# WORLD CUP WINNER PREDICTION - DATA VISUALIZATION (Matplotlib + Seaborn)
# File: train.csv
# =============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------------------
# STEP 0 : LOAD DATA
# -------------------------------------------------------------
TRAIN_PATH = 'train_cleaned.csv'      # apna path yahan set karo
df = pd.read_csv(TRAIN_PATH)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

TARGET = 'winner'
num_cols = df.select_dtypes(include=np.number).columns.tolist()

# -------------------------------------------------------------
# STEP 1 : HISTOGRAM (distribution of numeric columns)
# -------------------------------------------------------------
plt.figure()
sns.histplot(df['fifa_points'], bins=25, kde=True, color='skyblue')
plt.title('Distribution of FIFA Points')
plt.savefig('01_hist_fifa_points.png')
plt.close()

plt.figure()
sns.histplot(df['avg_player_rating'], bins=25, kde=True, color='orange')
plt.title('Distribution of Average Player Rating')
plt.savefig('02_hist_player_rating.png')
plt.close()

# -------------------------------------------------------------
# STEP 2 : COUNTPLOT (target class balance + categorical)
# -------------------------------------------------------------
plt.figure()
sns.countplot(x=TARGET, data=df, hue=TARGET, palette='Set2', legend=False)
plt.title('Winner Class Distribution (Target Balance)')
plt.xlabel('Winner (1 = Yes, 0 = No)')
plt.savefig('03_count_target.png')
plt.close()

plt.figure()
sns.countplot(y='confederation', data=df, order=df['confederation'].value_counts().index,
              hue='confederation', palette='viridis', legend=False)
plt.title('Number of Teams by Confederation')
plt.savefig('04_count_confederation.png')
plt.close()

# -------------------------------------------------------------
# STEP 3 : BOXPLOT (numeric spread across target / category)
# -------------------------------------------------------------
plt.figure()
sns.boxplot(x=TARGET, y='fifa_points', data=df, hue=TARGET, palette='coolwarm', legend=False)
plt.title('FIFA Points by Winner Outcome')
plt.savefig('05_box_fifapoints_target.png')
plt.close()

plt.figure()
sns.boxplot(x='confederation', y='avg_player_rating', data=df,
            hue='confederation', palette='Set3', legend=False)
plt.title('Player Rating by Confederation')
plt.xticks(rotation=30)
plt.savefig('06_box_rating_confederation.png')
plt.close()

# -------------------------------------------------------------
# STEP 4 : VIOLIN PLOT (distribution shape by category)
# -------------------------------------------------------------
plt.figure()
sns.violinplot(x=TARGET, y='market_value_million_eur', data=df,
               hue=TARGET, palette='muted', legend=False)
plt.title('Market Value Distribution by Winner Outcome')
plt.savefig('07_violin_marketvalue_target.png')
plt.close()

# -------------------------------------------------------------
# STEP 5 : BAR CHART (aggregated comparisons)
# -------------------------------------------------------------
win_rate_by_confed = df.groupby('confederation')[TARGET].mean().sort_values(ascending=False)
plt.figure()
sns.barplot(x=win_rate_by_confed.values, y=win_rate_by_confed.index,
            hue=win_rate_by_confed.index, palette='rocket', legend=False)
plt.title('Win Rate by Confederation')
plt.xlabel('Win Rate')
plt.savefig('08_bar_winrate_confederation.png')
plt.close()

top10_rank = df.sort_values('fifa_rank').head(10)
plt.figure()
sns.barplot(x='fifa_points', y='team_name', data=top10_rank,
            hue='team_name', palette='crest', legend=False)
plt.title('Top 10 Ranked Teams - FIFA Points')
plt.savefig('09_bar_top10_teams.png')
plt.close()

# -------------------------------------------------------------
# STEP 6 : SCATTER PLOT (relationship between numeric variables)
# -------------------------------------------------------------
plt.figure()
sns.scatterplot(x='fifa_rank', y='fifa_points', data=df, hue=TARGET, alpha=0.7, palette='coolwarm')
plt.title('FIFA Rank vs FIFA Points (colored by Winner)')
plt.savefig('10_scatter_rank_points.png')
plt.close()

plt.figure()
sns.scatterplot(x='avg_player_rating', y='market_value_million_eur', data=df,
                 hue=TARGET, alpha=0.7, palette='viridis')
plt.title('Player Rating vs Market Value')
plt.savefig('11_scatter_rating_marketvalue.png')
plt.close()

# -------------------------------------------------------------
# STEP 7 : PAIRPLOT (multiple relationships at once)
# -------------------------------------------------------------
subset_cols = ['fifa_points', 'avg_player_rating', 'market_value_million_eur',
               'win_rate_last_year', TARGET]
sns.pairplot(df[subset_cols], hue=TARGET, palette='husl', diag_kind='kde')
plt.savefig('12_pairplot_subset.png')
plt.close()

# -------------------------------------------------------------
# STEP 8 : CORRELATION HEATMAP
# -------------------------------------------------------------
plt.figure(figsize=(16, 12))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap - All Numeric Features')
plt.tight_layout()
plt.savefig('13_heatmap_correlation.png')
plt.close()

# Focused heatmap: correlation with target only
plt.figure(figsize=(6, 10))
target_corr = df[num_cols].corr()[[TARGET]].sort_values(by=TARGET, ascending=False)
sns.heatmap(target_corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation with Target (winner)')
plt.tight_layout()
plt.savefig('14_heatmap_target_correlation.png')
plt.close()

# -------------------------------------------------------------
# STEP 9 : KDE PLOT (density comparison by class)
# -------------------------------------------------------------
plt.figure()
sns.kdeplot(df[df[TARGET] == 1]['recent_form_score'], label='Winner', fill=True, alpha=0.4)
sns.kdeplot(df[df[TARGET] == 0]['recent_form_score'], label='Non-Winner', fill=True, alpha=0.4)
plt.title('Recent Form Score Density: Winner vs Non-Winner')
plt.legend()
plt.savefig('15_kde_formscore_target.png')
plt.close()

# -------------------------------------------------------------
# STEP 10 : PIE CHART (proportion / share)
# -------------------------------------------------------------
plt.figure()
confed_counts = df['confederation'].value_counts()
plt.pie(confed_counts, labels=confed_counts.index, autopct='%1.1f%%',
        colors=sns.color_palette('pastel'))
plt.title('Team Share by Confederation')
plt.savefig('16_pie_confederation.png')
plt.close()

# -------------------------------------------------------------
# STEP 11 : LINE / TREND (rank vs average form)
# -------------------------------------------------------------
rank_bins = pd.cut(df['fifa_rank'], bins=10)
form_by_rank_bin = df.groupby(rank_bins, observed=True)['recent_form_score'].mean()
plt.figure()
form_by_rank_bin.plot(kind='line', marker='o', color='teal')
plt.title('Average Recent Form Score across FIFA Rank Bins')
plt.xlabel('FIFA Rank Bin')
plt.ylabel('Avg Recent Form Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('17_line_form_by_rankbin.png')
plt.close()

# -------------------------------------------------------------
# STEP 12 : FACETGRID (multiple subplots by category)
# -------------------------------------------------------------
g = sns.FacetGrid(df, col='host_advantage', height=5)
g.map(sns.histplot, 'win_rate_last_year', bins=20, color='steelblue')
g.set_titles('Host Advantage: {col_name}')
g.savefig('18_facetgrid_winrate_hostadvantage.png')
plt.close()

# -------------------------------------------------------------
# STEP 13 : JOINTPLOT (scatter + distribution combined)
# -------------------------------------------------------------
sns.jointplot(x='possession_avg', y='passing_accuracy', data=df, kind='hex', color='darkorange')
plt.savefig('19_jointplot_possession_passing.png')
plt.close()

# -------------------------------------------------------------
# STEP 14 : DASHBOARD-STYLE SUBPLOTS (summary view)
# -------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['fifa_points'], bins=20, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('FIFA Points Distribution')

sns.boxplot(x=TARGET, y='goals_scored_avg', data=df, hue=TARGET,
            ax=axes[0, 1], palette='Set3', legend=False)
axes[0, 1].set_title('Goals Scored Avg by Winner Outcome')

sns.scatterplot(x='shots_per_game', y='avg_player_rating', data=df,
                 hue=TARGET, ax=axes[1, 0], alpha=0.6, palette='coolwarm')
axes[1, 0].set_title('Shots per Game vs Player Rating')

win_rate_top = df.groupby('confederation')[TARGET].mean().sort_values(ascending=False).head(5)
sns.barplot(x=win_rate_top.values, y=win_rate_top.index,
            hue=win_rate_top.index, ax=axes[1, 1], palette='mako', legend=False)
axes[1, 1].set_title('Top 5 Confederations by Win Rate')

plt.tight_layout()
plt.savefig('20_dashboard_subplots.png')
plt.close()

print("All visualizations generated and saved as PNG files in the working directory.")

All visualizations generated and saved as PNG files in the working directory.
